# Starter

Loads the clips, plays one, and draws a first map of all 60 clips from simple sound features.
This is a starting point, not an answer. Unzip the data into `data/` first (see the README).

In [ ]:
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

DATA = Path("data")
clips = pd.read_csv(DATA / "clips.csv", keep_default_na=False)
print(len(clips), "clips,", (clips.speaker != "").sum(), "labeled,", (clips.speaker == "").sum(), "to identify")
clips.head()

Listen to a clip.

In [ ]:
Audio(filename=DATA / "clips" / f"{clips.clip_id[0]}.wav")

A first fingerprint: the average and spread of 20 MFCCs per clip, a rough summary of the voice's tone.
Each dot is a clip, colored by speaker. Grey crosses are the clips to identify.

In [ ]:
def mfcc_summary(path):
    signal, rate = librosa.load(path, sr=None)
    mfcc = librosa.feature.mfcc(y=signal, sr=rate, n_mfcc=20)
    return np.concatenate([mfcc.mean(axis=1), mfcc.std(axis=1)])


features = np.array([mfcc_summary(DATA / "clips" / f"{clip_id}.wav") for clip_id in clips.clip_id])
points = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(features))

fig, ax = plt.subplots(figsize=(8, 6))
speakers = sorted(s for s in clips.speaker.unique() if s)
colors = plt.get_cmap("tab10")
for i, speaker in enumerate(speakers):
    mask = (clips.speaker == speaker).to_numpy()
    ax.scatter(points[mask, 0], points[mask, 1], color=colors(i), label=speaker, s=40)
unknown = (clips.speaker == "").to_numpy()
ax.scatter(points[unknown, 0], points[unknown, 1], color="grey", marker="x", label="unknown", s=40)
for (x, y), clip_id in zip(points[unknown], clips.clip_id[unknown]):
    ax.annotate(clip_id.removeprefix("clip_"), (x, y), fontsize=7, color="grey")
ax.set_xlabel("PCA 1")
ax.set_ylabel("PCA 2")
ax.legend(fontsize=8, ncol=2)
ax.set_title(f"{len(clips)} clips, simple MFCC features")
plt.show()

Speakers that form tight clusters here are easy to tell apart, and overlapping ones are not.
Better features, better models and a better map are up to you.